
# Phase guess using optimal transport

For intricate targets, the combination of a linear and a quadratic phase term we applied
before is not sufficient to ensure a vortex-free result. Recent work employs optimal
transport methods to find an accurate initial phase guess, leading to improved
convergence:

    A. Torchylo, H. Swan, L. Tellez and J. M. Hogan, *A fast, large-scale optimal
    transport algorithm for holographic beam shaping*,
    [arXiv:2512.19072 (2025)](https://arxiv.org/abs/2512.19072).

    H. Swan, A. Torchylo, M. J. Van de Graaff, J. Rudolph and J. M. Hogan,
    *High-fidelity holographic beam shaping with optimal transport and phase
    diversity*, [Opt. Express 33, 6290 (2025)](https://doi.org/10.1364/OE.540901).

HoloGradPy implements the scalable version, which does not require any down-sampling of
the target or the SLM beam profile.


In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image

import hologradpy
from hologradpy.analysis.error_metrics import (
    DEFAULT_INTENSITY_METRICS,
    efficiency_metric,
    evaluate_metrics,
)
from hologradpy.holography.phase_retrieval import (
    PixelwisePhaseRetriever,
    OptimalTransportPhaseRetriever,
)
from hologradpy.holography.vortices import VortexDetector
from hologradpy.holography.vortices.visualizer import (
    NEGATIVE_COLOR,
    POSITIVE_COLOR,
)
from hologradpy.optics.complex_amplitude import ComplexAmplitude, FieldGeometry
from hologradpy.optics.modules.slm_fields import PixelwiseSLMField
from hologradpy.optics.modules.virtual_slms import VirtualSLM
from hologradpy.optics.systems import SLMFFT
from hologradpy.profiles.amplitude import gaussian_beam_intensity
from hologradpy.profiles.phase import gaussian_phase_guess
from hologradpy.roi import ROI
from hologradpy.utils import get_device, gpu_to_numpy, to_canvas
from hologradpy.visualizer import INTENSITY_CMAP, image_grid

device = get_device(verbose=True)

WAVELENGTH = 670e-9
FOCAL_LENGTH = 250e-3
BEAM_RADIUS = 4e-3
SLM_RESOLUTION = (1024, 1280)
SLM_PIXEL_SIZE = (12.5e-6, 12.5e-6)

REGULARIZATION = 1e-3
SINKHORN_ITERATIONS = 400

SIGNAL_MARGIN = 40
LBFGS_ITERATIONS = 200

# Vortices are only counted where the target is lit above this fraction of its
# peak. One out in the dark surround costs nothing and would flatter both runs.
VORTEX_THRESHOLD = 0.2

## Setting up a model of the optical system



In [ ]:
slm_geometry = FieldGeometry(
    resolution=SLM_RESOLUTION,
    pixel_size=torch.tensor(list(SLM_PIXEL_SIZE), device=device),
    wavelength=torch.tensor(WAVELENGTH, device=device),
)
slm_grid = slm_geometry.get_spatial_grid()
slm_field = ComplexAmplitude.from_geometry(
    slm_geometry,
    data=gaussian_beam_intensity(*slm_grid, beam_radius=BEAM_RADIUS).sqrt() + 0j,
)


def build_model() -> SLMFFT:
    """A fresh model, so the runs below start from the same blank SLM."""
    model = SLMFFT(
        input_geometry=slm_field.geometry,
        virtual_slm=VirtualSLM(phase_scaling=1.0),
        slm_field=PixelwiseSLMField(slm_field),
        focal_length=FOCAL_LENGTH,
        padded_resolution=(2048, 2048),
    )
    model()
    return model

## Defining the target

We use something difficult to show the advantage of optimal transport.




In [ ]:
model = build_model()
output_resolution = tuple(int(size) for size in model[-1].resolution_out)

targets = Path(hologradpy.__file__).parents[1] / "targets"
picture = np.asarray(Image.open(targets / "duke.jpg").convert("L"), dtype=np.float64)
picture /= picture.max()

target_intensity = torch.as_tensor(
    to_canvas(picture, output_resolution), dtype=torch.float32, device=device
)
lit = np.ones(np.array(picture.shape) + 2 * SIGNAL_MARGIN, dtype=bool)
signal_region = torch.as_tensor(
    to_canvas(lit, output_resolution), dtype=torch.bool, device=device
)

# Every figure below shows the target's own corner of the output plane rather than the
# mostly empty canvas it sits on.
frame = ROI.detect(signal_region.to(torch.float32), threshold=0.5, pad=0)

# How wide the target is, which is what the analytic guess later spreads the beam to
# match. Size is all such a guess can know; where the light goes within it is what
# optimal transport adds.
camera_x, camera_y = model[-1].get_spatial_grid_output()
guess_radius = (
    picture.shape[1] * float(camera_x[0, 1] - camera_x[0, 0]) / 2,
    picture.shape[0] * float(camera_y[1, 0] - camera_y[0, 0]) / 2,
)


def framed(image: torch.Tensor | np.ndarray) -> np.ndarray:
    return frame.crop(gpu_to_numpy(image))


figure = image_grid(
    [[framed(target_intensity)]],
    titles=["target"],
    cmap=INTENSITY_CMAP,
    colorbar_label="intensity [a. u.]",
    column_width=4.0,
).build()

## Running the optimal transport optimization

The output generated by the optimal transport phase guess is already close to the
target.




In [ ]:
guess_model = build_model()
guess = OptimalTransportPhaseRetriever(
    guess_model, target_intensity.clone(), regularization=REGULARIZATION
)
guess_phase = guess.retrieve_phase(SINKHORN_ITERATIONS)
with torch.no_grad():
    guess_intensity = gpu_to_numpy(guess_model().intensity.squeeze())

figure = image_grid(
    [[gpu_to_numpy(guess_phase) % (2 * np.pi), framed(guess_intensity)]],
    titles=["optimal transport phase guess", "corresponding output"],
    cmap=["magma", INTENSITY_CMAP],
    column_width=3.2,
).build()

## Tuning the guess with L-BFGS

Optimizing the phase pattern using a gradient-based method, starting from the optimal
transport guess.



In [ ]:
metrics = DEFAULT_INTENSITY_METRICS + (
    efficiency_metric(model.incident_power(), model.output_pixel_area()),
)

transport_record = OptimalTransportPhaseRetriever(
    model, target_intensity, regularization=REGULARIZATION
).retrieve(SINKHORN_ITERATIONS, name="optimal transport", metrics=metrics)

transport_metrics = evaluate_metrics(
    metrics,
    gpu_to_numpy(signal_region),
    gpu_to_numpy(target_intensity),
    gpu_to_numpy(guess_model().intensity.squeeze().detach()),
)

refined = PixelwisePhaseRetriever(model, target_intensity, signal_region)
refined_record = refined.retrieve(
    LBFGS_ITERATIONS, name="l-bfgs", method="l-bfgs", metrics=metrics
)
seeded_records = [transport_record, refined_record]

with torch.no_grad():
    seeded_intensity = gpu_to_numpy(model().intensity.squeeze())

## Before and after




In [ ]:
figure = image_grid(
    [
        [
            framed(target_intensity),
            framed(guess_intensity),
            framed(seeded_intensity),
        ]
    ],
    titles=[
        "target",
        f"transport guess (rmse {transport_metrics['rmse'][-1]:.3f})",
        f"after L-BFGS (rmse {seeded_records[-1].metrics['rmse'][-1]:.3f})",
    ],
    cmap=INTENSITY_CMAP,
    colorbar_label="intensity [a. u.]",
    column_width=3.0,
).build()

## Starting with a quadratic phase guess




In [ ]:
lens_model = build_model()
lens_metrics = DEFAULT_INTENSITY_METRICS + (
    efficiency_metric(lens_model.incident_power(), lens_model.output_pixel_area()),
)
lens_model.virtual_slm.set_phase(
    gaussian_phase_guess(
        *slm_grid,
        input_beam_radius=(BEAM_RADIUS, BEAM_RADIUS),
        output_beam_radius=guess_radius,
        focal_length=FOCAL_LENGTH,
        wavenumber=2 * torch.pi / WAVELENGTH,
    ).to(torch.float32)
)

lens_guess_phase = gpu_to_numpy(lens_model.virtual_slm.get_phase().detach())
with torch.no_grad():
    lens_guess_intensity = gpu_to_numpy(lens_model().intensity.squeeze())

lens_records = [
    PixelwisePhaseRetriever(
        lens_model, target_intensity.clone(), signal_region.clone()
    ).retrieve(LBFGS_ITERATIONS, name="l-bfgs", method="l-bfgs", metrics=lens_metrics)
]

with torch.no_grad():
    lens_intensity = gpu_to_numpy(lens_model().intensity.squeeze())

figure = image_grid(
    [[lens_guess_phase % (2 * np.pi)]],
    titles=["quadratic phase on the SLM"],
    cmap="magma",
    colorbar_label="phase [rad]",
    column_width=4.0,
).build()

It spreads the beam over roughly the right area.



In [ ]:
figure = image_grid(
    [[framed(target_intensity), framed(lens_guess_intensity)]],
    titles=["target", "output from quadratic phase guess"],
    cmap=INTENSITY_CMAP,
    colorbar_label="intensity [a. u.]",
    column_width=4.0,
).build()

## Comparison

Although the rms errors are similar, the optimal transport guess converges to a
solution with a much higher efficiency and fewer optical vortices.
:class:`~hologradpy.holography.vortices.VortexDetector` finds vortices in each output
in regions where the target is brighter than 20% of the peak target intensity.



In [ ]:
def vortices_in(
    camera_model: SLMFFT,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Vortex positions as ``(x, y)`` pixel coordinates in the frame, and charges."""
    detector = VortexDetector(output_resolution, device=device)
    with torch.no_grad():
        detector.detect_vortices(
            camera_model(), target_intensity, threshold=VORTEX_THRESHOLD
        )
    if detector.number_of_vortices == 0:
        return np.empty(0), np.empty(0), np.empty(0)

    rows, columns = np.array([
        [int(row), int(column)] for row, column in detector.center_indices
    ]).T
    rows, columns = rows - frame.top_row, columns - frame.left_column
    charges = gpu_to_numpy(detector.charges).reshape(-1)
    inside = (
        (charges != 0)
        & (rows >= 0)
        & (rows < frame.height)
        & (columns >= 0)
        & (columns < frame.width)
    )
    return columns[inside], rows[inside], charges[inside]


seeded_vortices = vortices_in(model)
lens_vortices = vortices_in(lens_model)


def report(label, records):
    for record in records:
        row = {key: values[-1] for key, values in record.metrics.items() if values}
        efficiency = next(
            (value for key, value in row.items() if "efficiency" in key), float("nan")
        )
        print(
            f"  {label:>10} / {record.name:<20} "
            f"rmse {row.get('rmse', float('nan')):.4f}   "
            f"efficiency {efficiency:6.1%}"
        )


print(
    f"  {'seeded':>10} / {'optimal transport':<20} "
    f"rmse {transport_metrics['rmse'][-1]:.4f}   "
    f"efficiency {transport_metrics['signal efficiency'][-1]:6.1%}"
)
report("seeded", [refined_record])
report("from lens", lens_records)

comparison = image_grid(
    [[framed(target_intensity), framed(seeded_intensity), framed(lens_intensity)]],
    titles=[
        "target",
        f"optimal transport guess ({len(seeded_vortices[0])} vortices)",
        f"quadratic phase guess ({len(lens_vortices[0])} vortices)",
    ],
    cmap=INTENSITY_CMAP,
    colorbar_label="intensity [a. u.]",
    column_width=4.0,
)
for cell, (vortex_x, vortex_y, charges) in (
    ("1", seeded_vortices),
    ("2", lens_vortices),
):
    for sign, colour, label in (
        (1, POSITIVE_COLOR, "charge +1"),
        (-1, NEGATIVE_COLOR, "charge -1"),
    ):
        keep = np.sign(charges) == sign
        if not keep.any():
            continue

        comparison = comparison.draw_points(
            cell,
            vortex_x[keep],
            vortex_y[keep],
            marker="o",
            color=colour,
            size=4,
            edgecolor=colour,
            label=label,
            legend=True,
            markerfacecolor="none",
            markeredgewidth=0.9,
        )
figure = comparison.build()